In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return "Email sent"

In [22]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str
    
agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval"
        )
    ]
)

In [23]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response.")],
        "email": "Hi Filipe, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John"
    },
    config=config
)

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Please read my email and send a response.
================================== Ai Message ==================================
Tool Calls:
  read_email (call_tqtilRAqtAt8Iy6VDHQvqrwf)
 Call ID: call_tqtilRAqtAt8Iy6VDHQvqrwf
  Args:
================================= Tool Message =================================
Name: read_email

Hi Filipe, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John
================================== Ai Message ==================================
Tool Calls:
  send_email (call_WHuehJV5MOh5k8xIAPplR9rb)
 Call ID: call_WHuehJV5MOh5k8xIAPplR9rb
  Args:
    body: Subject: Re: Meeting tomorrow

Hi John,

No problem—thanks for the heads up. I’m flexible on the reschedule. What time works best for you tomorrow? If it helps, I can propose a few options (morning, early afternoon, or late afternoon), or you can suggest a time that suits you.

Best regards,
Filipe


In [16]:
print(response["__interrupt__"])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for letting me know. I’m happy to reschedule. Could we move the meeting to tomorrow at 10:00 AM or 3:00 PM? If neither works, please share a couple of times that fit your schedule and I’ll adjust.\n\nBest regards,\nFilipe'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for letting me know. I’m happy to reschedule. Could we move the meeting to tomorrow at 10:00 AM or 3:00 PM? If neither works, please share a couple of times that fit your schedule and I’ll adjust.\\n\\nBest regards,\\nFilipe'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='0be8c51e0e3d99b7f516f0207b964ed7')]


In [17]:
print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])

Hi John,

No problem—thanks for letting me know. I’m happy to reschedule. Could we move the meeting to tomorrow at 10:00 AM or 3:00 PM? If neither works, please share a couple of times that fit your schedule and I’ll adjust.

Best regards,
Filipe


## Approve

In [12]:
from langgraph.types import Command
from pprint import pprint

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': "Hi Filipe, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John',
 'messages': [HumanMessage(content='Please read my email and send a response.', additional_kwargs={}, response_metadata={}, id='23991ed0-2a4d-41d1-bf97-67afb1712a59'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 467, 'prompt_tokens': 157, 'total_tokens': 624, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cul2W7i1EwXm6cDi7cQmiSAPaHkeS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b8fbe-d932-7ce1-a24e-c2693e38be9c-0', tool_calls=[{'name': 'read_email', 'args': {}, 'id': 'call_N

## Reject

In [18]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Filipe"
                }
            ]
        }
    ),
    config=config
)

In [19]:
pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'I’m '
                                                                          'happy '
                                                                          'to '
            

In [20]:
print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])

Hi John,

No problem—thanks for letting me know. I’m happy to reschedule. Could we move the meeting to tomorrow at 10:00 AM or 3:00 PM? If neither works, please share a couple of times that fit your schedule and I’ll adjust.

Your merciful leader,
Filipe


## Edit

In [24]:
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {
                            "body": "This is the last straw, you're fired!"
                        }
                    }
                }
            ]
        }
    ),
    config=config
)

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'Thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads-up. '
                                                                          'I '
                                                                          'understand '
                                                                          'you’ll '
                                                                          'be '
                                                                          'late '
           